In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np

BATCH_SIZE = 128
LEARNING_RATE = 1e-3
EPOCHS = 20
NUM_CLUSTERS = 100 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEMPERATURE = 0.1

In [3]:
class ContrastiveTransformations:
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size=32),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
        ])

    def __call__(self, x):
        return [self.transform(x), self.transform(x)]

train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, 
                                 transform=ContrastiveTransformations())
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

100%|██████████| 169M/169M [00:12<00:00, 13.5MB/s] 


In [6]:
class SemanticClusteringModel(nn.Module):
    def __init__(self, backbone, feature_dim=512, cluster_dim=100):
        super(SemanticClusteringModel, self).__init__()
        self.backbone = backbone
        self.backbone.fc = nn.Identity()

        self.projection_head = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, 128)
        )

        self.cluster_head = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, cluster_dim),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        features = self.backbone(x)
        projections = self.projection_head(features)
        clusters = self.cluster_head(features)
        return projections, clusters

resnet = models.resnet18(weights=None)
model = SemanticClusteringModel(resnet, feature_dim=512, cluster_dim=NUM_CLUSTERS).to(DEVICE)

In [4]:
def info_nce_loss(features, batch_size, temperature=0.1):
    # features: (2 * BATCH_SIZE, 128)
    labels = torch.cat([torch.arange(batch_size) for i in range(2)], dim=0)
    labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float().to(DEVICE)

    features = F.normalize(features, dim=1)
    similarity_matrix = torch.matmul(features, features.T)

    # Loại bỏ các đường chéo (tự tương quan)
    mask = torch.eye(labels.shape[0], dtype=torch.bool).to(DEVICE)
    labels = labels[~mask].view(labels.shape[0], -1)
    similarity_matrix = similarity_matrix[~mask].view(similarity_matrix.shape[0], -1)

    # Lấy các cặp dương (positive samples)
    positives = similarity_matrix[labels.bool()].view(labels.shape[0], -1)
    # Lấy các cặp âm (negatives)
    negatives = similarity_matrix[~labels.bool()].view(similarity_matrix.shape[0], -1)

    logits = torch.cat([positives, negatives], dim=1)
    logits /= temperature

    target = torch.zeros(labels.shape[0], dtype=torch.long).to(DEVICE)
    return F.cross_entropy(logits, target)

def cluster_loss(p_out, q_out):
    # p_out, q_out: Xác suất dự đoán cụm của 2 view (Batch_size, 100)
    
    # 1. Tính xác suất trung bình của cả Batch
    avg_probs = (p_out.mean(0) + q_out.mean(0)) / 2
    
    # 2. Marginal Entropy (Càng cao càng tốt -> Loss càng âm càng tốt)
    # Giúp các cụm được sử dụng đồng đều
    me_loss = - torch.sum(avg_probs * torch.log(avg_probs + 1e-8))
    
    # 3. Conditional Entropy (Càng thấp càng tốt -> Loss càng nhỏ càng tốt)
    # Giúp model tự tin vào việc gán cụm cho mỗi ảnh
    cond_ent = - (torch.sum(p_out * torch.log(p_out + 1e-8), dim=1).mean() + 
                  torch.sum(q_out * torch.log(q_out + 1e-8), dim=1).mean()) / 2
    
    # Trả về: Thấp (Tự tin) - Cao (Đa dạng)
    return cond_ent - me_loss

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for (view1, view2), _ in train_loader:
        view1, view2 = view1.to(DEVICE), view2.to(DEVICE)
        
        proj1, clust1 = model(view1)
        proj2, clust2 = model(view2)
        
        features = torch.cat([proj1, proj2], dim=0)
        loss_con = info_nce_loss(features, BATCH_SIZE, TEMPERATURE)
        loss_clu = cluster_loss(clust1, clust2)
        
        loss = loss_con + loss_clu
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_loader):.4f}")